# E1 · W3 · Kaggle run 05 — phân tích baseline A trên val (CPU)
**Mục đích:** learning curves, P/R/F1 theo ngưỡng conf (chọn trên val), recall theo cỡ bbox và theo video, nhầm lớp, false positive trên ảnh âm tính (issue #19). Không dùng test.

**Trước khi chạy:**
1. *Add Input*: `e1-w3-tools` (bản có `w3_analyze_baseline.py`), output notebook `e1-w3-run03b`, output notebook `e1-w3-run04`.
2. *Settings*: Accelerator **None**, Internet **On**.
3. Điền `COMMIT`.

**Sau khi chạy:** *Save Version → Save & Run All*. Tải `w3_analysis.zip` → giải nén vào thư mục kết quả run 05 (xem hướng dẫn local).

In [ ]:
COMMIT = ""   # <-- SHA day du 40 ky tu

In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
cards = [p for p in glob.glob("/kaggle/input/**/dataset_card.json", recursive=True)
         if os.path.exists(os.path.join(os.path.dirname(p), "data.yaml"))]
assert len(cards) == 1, f"Can 1 bo du lieu YOLO (output run 03b), thay {cards}"
DATA_DIR = os.path.dirname(cards[0])
MANIFEST = find("yolo_manifest.csv.gz")
EVID = os.path.dirname(find("baseline_A_val_metrics.json"))
SCRIPT = find("w3_analyze_baseline.py")
print(DATA_DIR, MANIFEST, EVID, SCRIPT, sep="\n")

In [ ]:
!{sys.executable} {SCRIPT} --data-dir {DATA_DIR} --manifest {MANIFEST} --train-evidence {EVID} --out /kaggle/working --commit {COMMIT}

## Tự kiểm tra (phải in `PASS`)

In [ ]:
import pandas as pd
out = "/kaggle/working/w3_analysis"
s = json.load(open(f"{out}/analysis_summary.json")); print(s["status"])
print(json.dumps(s["stages"].get("operating_points"), indent=1))
print(pd.read_csv(f"{out}/recall_by_size.csv").to_string(index=False))
print("PASS" if all(v == "OK" for v in s["status"].values()) else "FAIL - gui analysis_summary.json cho Claude")
p = "/kaggle/working/w3_analysis.zip"; print(os.path.getsize(p) // 1024, "KB  sha256", sha(p))